# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
"""
Checking the fields this audit will test, before testing anything, per the auditing-signals
skill's first rule.

""

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

for col in ["impressions_90d", "clicks_90d", "search_volume", "word_count"]:
    s = df[col].dropna()
    print(f"{col:16s}  mean={s.mean():>10.1f}  median={s.median():>8.1f}  "
          f"max={s.max():>10.0f}  mean/median ratio={s.mean()/max(s.median(),1):>6.1f}")

In [ ]:
'''

Heavy tails, confirmed: impressions_90d (mean 5,200 vs median 731 — mean is ~7x the median),
clicks_90d (mean 16 vs median 1 — 16x), and search_volume (mean 159 vs median 10 — 16x) are
all severely right-skewed: a handful of giant pages/keywords, a long tail of tiny ones.
word_count is comparatively well-behaved (mean 3,108 vs median 2,877 — close to 1x).

What this changes below: for the skewed columns, a raw Pearson correlation or a plain mean
comparison would be dominated by outliers. So every test below uses medians and quartile
buckets instead of means where the field is heavy-tailed.

'''

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
'''
Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.
'''

In [ ]:
'''
### Test 1 — "Longer content earns more impressions"

Bucket: word_count_tier. Outcome: impressions_90d (median, since it's heavy-tailed).

'''

In [ ]:
t1 = df.groupby("word_count_tier")["impressions_90d"].agg(["median", "mean", "count"])
print(t1.round(1))

In [ ]:
'''
Verdict: CONFIRMED.

Median impressions climb cleanly with word count: <1000=4 (n=973), 1000-2000=172 (n=3,780),
2000-3500=997 (n=11,263), 3500+=1,340 (n=6,285). Every bucket clears the ~50-row floor by a
wide margin, and the direction is monotonic across all four tiers.

Caveat: this shows word count and impressions move together, not that length causes traffic —
longer pieces may also be the ones FlyRank's writers invest more research/keywords into. Worth
a follow-up test controlling for content_type before treating length itself as the lever.
'''

In [ ]:
'''
### Test 2 — "Higher search volume keywords bring more clicks"

Spearman (rank) correlation, since both search_volume and clicks_90d are heavy-tailed — plus
a quartile-bucketed table.

'''

In [ ]:
valid = df.dropna(subset=["search_volume"])
spearman_r = valid["search_volume"].corr(valid["clicks_90d"], method="spearman")
print("Spearman r (search_volume vs clicks_90d):", round(spearman_r, 3), " n=", len(valid))

sv_quartile = pd.qcut(valid["search_volume"].rank(method="first"), q=4,
                       labels=["q1_low", "q2", "q3", "q4_high"])
t2 = valid.groupby(sv_quartile, observed=True)["clicks_90d"].agg(["median", "mean", "count"])
print(t2.round(2))

In [ ]:
'''
Verdict: FALSE.

Spearman r = -0.068 — essentially no relationship, and what tiny signal exists points the
wrong way. The bucketed table confirms it: median clicks_90d is stuck at 1.0 across every
search-volume quartile, and the mean actually falls slightly from the lowest quartile (18.8)
to the highest (15.3). All four buckets have n=6,883, well clear of the floor.

Likely explanation, not proven: search_volume looks like it's a property of one target
keyword, while clicks_90d is the page's total organic clicks across however many keywords it
actually ranks for — a real mismatch between what's measured on each side.

'''

In [ ]:
'''
### Test 3 — "AI-generated content underperforms in engagement"

Bucket: provider_used (google vs openai). Outcome: engagement_rate.

'''

In [ ]:
t3 = df.groupby("provider_used")["engagement_rate"].agg(["median", "mean", "count"])
print(t3.round(2))

for p in ["google", "openai"]:
    sub = df.loc[df["provider_used"] == p, "engagement_rate"]
    zero_share = (sub == 0).mean()
    nonzero_median = sub[sub > 0].median()
    print(f"{p:8s} n={len(sub):5d}  zero_share={zero_share:.3f}  median_when_nonzero={nonzero_median:.2f}")

In [ ]:
'''
Verdict: MIXED.

By the overall mean, openai looks worse (1.61 vs google's 2.40), which would seem to confirm
the claim. But both n's clear the floor (7,364 vs 1,198) and the overall median is tied at 0.0
for both — so that mean gap is another heavy-tail artifact, not a typical-page difference.

Splitting it open: openai content has a noticeably higher zero-engagement share (84.2% vs
google's 70.9%) — it's less likely to get any engagement at all. But among pages that DO get
engagement, openai's median is actually slightly higher (5.66 vs 4.66) — when it lands, it
lands about as well or better. So the honest read is two different findings tangled into one
popular claim: a reach gap (real, favors google) and an intensity gap (doesn't exist, if
anything favors openai) — "AI content underperforms" as a single blanket claim is MIXED.

'''

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
'''
Flag: quick-win. The assumption behind it: a page already sitting in striking distance
(position_tier == "striking", roughly positions 11-20) that targets a real-demand keyword
(search_volume) is a good candidate to push toward page 1. This is the same search_volume
field Test 2 already found unreliable at the whole-dataset level; here I'm testing the
narrower, flag-specific claim: does it hold within the striking-distance slice the flag
actually fires on?

'''

In [ ]:
striking = df[df["position_tier"] == "striking"].dropna(subset=["search_volume"])
print("n striking-distance pages with search_volume:", len(striking))

sv_quartile = pd.qcut(striking["search_volume"].rank(method="first"), q=4,
                       labels=["q1_low", "q2", "q3", "q4_high"])
t_flag = striking.groupby(sv_quartile, observed=True).agg(
    median_impressions=("impressions_90d", "median"),
    median_clicks=("clicks_90d", "median"),
    n=("content_id", "count"),
)
print(t_flag)

In [ ]:
'''
Verdict: FALSE.

Within striking-distance pages specifically, search_volume quartile shows no relationship to
how much traffic the page already gets: median clicks_90d is flat at 1.0 in every single
quartile, and median impressions_90d bounces around (940 -> 1,011 -> 826 -> 918) with no
monotonic trend. Every quartile clears the floor easily (n=1,764-1,765). The quick-win flag's
assumption is not supported by this field in this snapshot.

This lines up with Test 2's finding: if search_volume describes one keyword but the page's
traffic reflects many, then a flag that leans on search_volume as a proxy for "how big is this
opportunity" is standing on a field that doesn't actually move with the outcome it predicts.

'''

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
'''

A content team can trust word count as a real (if likely confounded) length signal, but should
NOT use the search_volume field alone to size an opportunity or justify the quick-win flag —
it shows essentially no relationship to the traffic a page actually gets, at whole-dataset or
striking-distance level, so any prioritization built on it needs a second, page-level demand
signal before real budget follows it. And "AI content underperforms" needs to be split into
two separate claims before acting on it: it's genuinely less likely to get any engagement at
all (a reach problem, worth investigating), but when it does engage, it performs about as well
or slightly better than google-provider content (not an intensity problem) — treating it as
one blanket quality issue would misdiagnose which lever to pull.

'''

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.